# 04 · Semantic diagnostics

**Question:** Why do compact semantic relationships transfer more plausibly than raw embedding coordinates?

Frozen Qwen embeddings summarize each comment and its supplied examples. The joint DeBERTa probe adds rule/support entailment relationships. Neither encoder is fine-tuned on competition labels.

In [1]:
import os
from pathlib import Path
import pandas as pd
from IPython.display import display
from jigsaw_rules.review import public_evidence
from jigsaw_rules.runtime import environment

root = Path(os.environ.get("JIGSAW_ROOT", Path.cwd())).resolve()
if root.name == "notebooks":
    root = root.parent
baseline = public_evidence(root, "baseline")
semantic = public_evidence(root, "semantic")
assert baseline["training_sha256"] == semantic["training_sha256"]
print("Competition evidence | 2,029 rows | two labeled rules")
print("Aggregate checksums verified. This notebook performs no model fitting.")
protocols = {"seen_rule": "Familiar rules", "heldout_rule": "Held-out rule"}

def metric_table(records, heldout=False):
    return pd.DataFrame([{"Representation": r["model"], "Validation": protocols[r["protocol"]],
        "Rule macro AUC": r["metrics"]["rule_macro_auc"],
        "Log loss": r["metrics"]["log_loss"], "Brier": r["metrics"]["brier"],
        "Average precision": r["metrics"]["average_precision"]}
        for r in records if not heldout or r["protocol"] == "heldout_rule"]).round(4)


Competition evidence | 2,029 rows | two labeled rules
Aggregate checksums verified. This notebook performs no model fitting.


In [2]:
import sys
sys.path.insert(0, str(root / "scripts"))
from build_research_report import display_figure
from jigsaw_rules.features import feature_evidence
from jigsaw_rules.research import research_evidence
from jigsaw_rules.diagnostics import diagnostic_evidence
from jigsaw_rules.pairs import pairs_evidence
from jigsaw_rules.robustness import robustness_evidence
from jigsaw_rules.gate import feature_gate
from jigsaw_rules.instructions import instruction_evidence

controls = feature_evidence(root)
research = research_evidence(root)
sensitivity = diagnostic_evidence(root)
pairs = pairs_evidence(root)
robustness = robustness_evidence(root)
assert all(item is not None for item in (controls, research, sensitivity, pairs, robustness))
gate = feature_gate(root)
instructions = instruction_evidence(root)
print("Verified research runs:", gate["studies"])


Verified research runs: {'research': '9ec008d506b0bc64a717', 'sensitivity': 'bce07fd60bc7543fca49', 'pairs': '537cf2213c813b8ebd4b', 'robustness': 'a09455ec14e9b3d6ab61', 'instructions': '96bf69f42f9063e01a12'}


## Examine each observed policy
A mean over two policies can conceal deterioration on one. These tables preserve per-policy AUC and do not treat identical label-free predictions across validation protocols as independent replications.

In [3]:
records = baseline["results"] + semantic["results"]
records += [r for r in sensitivity["results"] if r["model"] in ["qwen_centroid", "qwen_mean", "qwen_maximum"]]
records += pairs["results"]
if instructions is not None:
    records += instructions["results"]
per_rule = pd.DataFrame([{"Representation": r["model"], "Rule": rule.split(":")[0], "ROC AUC": auc} for r in records if r["protocol"] == "heldout_rule" for rule, auc in r["metrics"]["per_rule_auc"].items()])
display(per_rule.pivot(index="Representation", columns="Rule", values="ROC AUC").round(4))

Rule,No Advertising,No legal advice
Representation,,
all_nli,0.5779,0.5425
comment_only,0.6495,0.5586
frozen_rule,0.4206,0.4801
frozen_rule_support,0.4122,0.4855
frozen_support,0.4268,0.5238
instruction_features,0.4522,0.5095
qwen_centroid,0.7288,0.5544
qwen_maximum,0.7135,0.5566
qwen_mean,0.7050,0.5523


## Probability quality and operating points
Calibration error uses ten equal-width bins. Precision, recall and F1 use a fixed diagnostic threshold of 0.5; threshold selection and calibration fitting are not claimed as completed. The temperatures of label-free scores are fixed transformations, not fitted calibrators.

In [4]:
diagnostics = pd.DataFrame([{"Representation": r["model"], "Pooled AUC": r["metrics"]["pooled_auc"], "Calibration error": r["metrics"]["ece_10_equal_width_bins"], "Precision@0.5": r["metrics"]["precision_at_0_5"], "Recall@0.5": r["metrics"]["recall_at_0_5"], "F1@0.5": r["metrics"]["f1_at_0_5"]} for r in records if r["protocol"] == "heldout_rule"])
display(diagnostics.round(4))

,Representation,Pooled AUC,Calibration error,Precision@0.5,Recall@0.5,F1@0.5
0,comment_only,0.6246,0.0366,0.5839,0.7051,0.6388
1,rule_examples,0.6317,0.0618,0.5693,0.8206,0.6722
2,semantic_margin,0.6221,0.0607,0.5893,0.5984,0.5938
3,semantic_classifier,0.4914,0.1679,0.5199,0.3928,0.4475
4,qwen_centroid,0.6271,0.0439,0.5917,0.5946,0.5931
5,qwen_maximum,0.6221,0.0607,0.5893,0.5984,0.5938
6,qwen_mean,0.6083,0.0598,0.5760,0.5917,0.5837
7,rule_nli,0.4849,0.0985,0.4838,0.5810,0.5280
8,support_nli,0.4433,0.1683,0.4891,0.6964,0.5746
9,all_nli,0.5244,0.0936,0.5077,0.6993,0.5883


## Compute and resumability
The original Qwen run encoded 1,875 unique texts; the broad study reused all verified embeddings. The NLI probe deduplicates comment–hypothesis pairs and records truncated pairs. Batches are hashed and published only after all files complete. The bounded AWS processing job checkpoints completed work to its own experiment prefix.

In [5]:
timing = semantic["timing"]
print("Original Qwen encoder:", timing["encoder"])
display(pd.Series({k: v for k, v in pairs["inference"].items() if k != "contract"}, name="NLI inference"))

Original Qwen encoder: {'unique_texts': 1875, 'encoded_texts': 1875, 'reused_texts': 0, 'truncated_rows': 1, 'encode_seconds': 983.6895047399812, 'peak_rss_gib': 3.854236602783203}


cache                         b8fef4f8c6bf3e0c473c
unique_pairs                                 11835
requested_pairs                              12174
original_inference_seconds             1658.648885
truncated_pairs                                 26
Name: NLI inference, dtype: object

## Interpretation and next research boundary
A positive-versus-negative centroid comparison uses the supplied task context directly and does not need hundreds of learned coefficients from a tiny fold. This is a plausible explanation for its transfer behavior, not a causal proof. Raw coordinates and many structural interactions are unstable across policies. General NLI features must earn their place through rule/support ablations, rather than being assumed superior because they use a transformer.

Further prompt or encoder searches would reuse already-inspected policies. Independent confirmation and a documented, legitimate source of broader rule coverage are the major unresolved avenues. [02 · Feature gate](02_baseline_and_review.ipynb) remains open. The [Kaggle notebook](../kaggle/submission.ipynb) remains the user's offline lexical-reference workflow; no automatic upload is performed.